# `02_all_variants.ipynb` — 12 feature-ablation variants on `EleutherAI/gpt-j-6b`

**Required input:** `kaggle_gptj_6b_dataset_full.json` (produced by `01_data_generation.ipynb`)

This notebook ASSUMES `kaggle_gptj_6b_dataset_full.json` is already in the working directory.
It does NOT re-generate data. If the file is missing, Stage 2 will print a clear error and stop.

## Stages

| # | Stage | Output (skipped on rerun if exists) |
|---|---|---|
| 1 | Setup + lazy LLM loader | (in memory) |
| 1.5 | Feature-extraction helpers (extract_mind, extract_extras) | (in memory) |
| 2 | **SKIPPED** — loads existing `kaggle_gptj_6b_dataset_full.json` | (errors loudly if missing) |
| 3 | F1–F10 feature extraction → augmented dataset | `kaggle_gptj_6b_dataset_with_features.json` |
| 4 | Train 12 MLP variants + Wikipedia held-out eval | `kaggle_gptj_6b_variant_<A..L>_best.pth` × 12 |
| 5+5b | Load 7 multi-task datasets + MiniLM scorer | (in memory) |
| 6 | Multi-task eval on all 12 variants × all 7 datasets | (in memory) |
| 7 | Consolidated dump | `kaggle_gptj_6b_all_variants_results.json` |

## After running

Paste `kaggle_gptj_6b_all_variants_results.json` back to the assistant for the cross-method comparison table.


In [ ]:
# =============================================================================
# BLOCK 0: PIP INSTALLS  +  (conditional) HuggingFace login for gated models
# =============================================================================
!pip install -q -U "transformers>=4.45" "tokenizers>=0.19" "accelerate>=0.30"
!pip install -q -U datasets spacy nltk scikit-learn tqdm sentence-transformers
!python -m spacy download en_core_web_sm

# -------- HuggingFace login (REQUIRED for gated models e.g. meta-llama/*) ----
# If EleutherAI/gpt-j-6b is gated you MUST:
#   1. Accept the model's license on huggingface.co (one-time per account).
#   2. Generate a User Access Token at huggingface.co/settings/tokens.
#   3. Paste the token when the cell below prompts.
# For non-gated models (gpt2, Qwen, etc.) the cell is a no-op.
_GATED_PREFIXES = ("meta-llama/", "google/gemma", "mistralai/")
if any("EleutherAI/gpt-j-6b".startswith(p) for p in _GATED_PREFIXES):
    from huggingface_hub import login
    login()   # opens an input box. Paste your HF token (starts with hf_...).
else:
    print("EleutherAI/gpt-j-6b is not gated -- skipping HF login.")


In [ ]:
# =============================================================================
# BLOCK 1 (STAGE 1): SETUP — imports, config, diagnostics dict, lazy LLM loader
# =============================================================================
import os, sys, gc, json, random, math, time, datetime, platform, traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MODEL_NAME       = "EleutherAI/gpt-j-6b"
MODEL_TAG        = "kaggle_gptj_6b"
TARGET_SAMPLES   = 1000       # per class
TOPK_FIRST_TOKEN = 4
WINDOWS          = 16
SEED             = 42
DTYPE            = torch.bfloat16 if torch.cuda.is_available() else torch.float32
LOGIT_LENS_LAYER = 14       # for F5 (DoLa-style)
MAX_GEN_NEW      = 48
EPOCHS           = 10
BATCH            = 32
LR               = 5e-4
WD               = 1e-5
TEST_FRAC        = 0.20

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PATH_DATASET_FULL     = f"{MODEL_TAG}_dataset_full.json"
PATH_DATASET_FEATURES = f"{MODEL_TAG}_dataset_with_features.json"
PATH_RESULTS          = f"{MODEL_TAG}_all_variants_results.json"
PATH_CHECKPOINT       = f"{MODEL_TAG}_datagen_checkpoint.json"

DIAG = {
    "schema_version": "all-variants-2.0",
    "model_tag":  MODEL_TAG,
    "model_name": MODEL_NAME,
    "config": {
        "target_samples_per_class": TARGET_SAMPLES,
        "topk_first_token": TOPK_FIRST_TOKEN, "windows": WINDOWS, "seed": SEED,
        "dtype": str(DTYPE), "logit_lens_layer": LOGIT_LENS_LAYER,
        "max_gen_new_tokens": MAX_GEN_NEW,
        "epochs": EPOCHS, "batch_size": BATCH,
        "lr": LR, "weight_decay": WD, "test_frac": TEST_FRAC,
    },
    "env": {
        "python":   sys.version.split()[0],
        "platform": platform.platform(),
        "torch":    torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device":   (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        "free_vram_gb":  (round(torch.cuda.mem_get_info()[0]/1e9, 2) if torch.cuda.is_available() else None),
    },
    "library_versions": {},
    "stage_timings_sec": {},
    "downstream_datasets": {},
    "errors": [],
}
for _mod in ["transformers", "datasets", "accelerate", "sklearn", "spacy", "nltk", "numpy"]:
    try:
        m = __import__(_mod)
        DIAG["library_versions"][_mod] = getattr(m, "__version__", "n/a")
    except Exception as e:
        DIAG["library_versions"][_mod] = f"IMPORT_FAILED: {e}"

_MODEL = {"tokenizer": None, "model": None, "nlp": None}

def need_llm():
    """Lazy-load the LLM + spacy. Idempotent."""
    if _MODEL["model"] is not None:
        return _MODEL
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import spacy, nltk
    nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
    print(f"Loading {MODEL_NAME} (dtype={DTYPE}) ...")
    _t0 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    # IMPORTANT: attn_implementation='eager' is required because Stage 3
    # extracts F1/F2/F6 from output_attentions=True, which SDPA does NOT
    # support. Without this, Qwen2.5 / Llama-2 / Falcon return None for
    # attentions and every F-feature silently falls back to 0.
    load_kwargs = dict(torch_dtype=DTYPE,
                       device_map="auto" if torch.cuda.is_available() else None,
                       attn_implementation="eager")
    try:
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    except (TypeError, ValueError) as _e:
        # Some custom models (e.g. older Falcon w/ trust_remote_code) reject
        # the kwarg. Retry without it; F1/F2/F6 may then be 0/NaN but the
        # other features and the rest of the pipeline still complete.
        print(f"[warn] retrying model load without attn_implementation: {_e}")
        load_kwargs.pop("attn_implementation", None)
        mdl = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    nlp = spacy.load("en_core_web_sm")
    DIAG["stage_timings_sec"]["model_load"] = round(time.perf_counter() - _t0, 2)
    DIAG["model_info"] = {
        "hidden_dim":    int(mdl.config.hidden_size),
        "n_layers":      int(mdl.config.num_hidden_layers),
        "n_heads":       int(getattr(mdl.config, "num_attention_heads", -1)),
        "vocab_size":    int(mdl.config.vocab_size),
        "max_pos_embed": int(getattr(mdl.config, "max_position_embeddings", -1)),
        "param_count_M": round(sum(p.numel() for p in mdl.parameters()) / 1e6, 2),
        "device":        str(next(mdl.parameters()).device),
    }
    print(f"✓ model loaded in {DIAG['stage_timings_sec']['model_load']}s   "
          f"hidden_dim={DIAG['model_info']['hidden_dim']}   "
          f"layers={DIAG['model_info']['n_layers']}")
    _MODEL["tokenizer"] = tok; _MODEL["model"] = mdl; _MODEL["nlp"] = nlp
    return _MODEL


print("=" * 80); print(f"STAGE 1 — SETUP — {MODEL_TAG}"); print("=" * 80)
print(f"Resume points:")
print(f"  Stage 2 -> {PATH_DATASET_FULL}      {'EXISTS — SKIP' if os.path.exists(PATH_DATASET_FULL) else 'will run'}")
print(f"  Stage 3 -> {PATH_DATASET_FEATURES}  {'EXISTS — SKIP' if os.path.exists(PATH_DATASET_FEATURES) else 'will run'}")
print(f"  Stage 7 -> {PATH_RESULTS}           {'EXISTS — overwrite' if os.path.exists(PATH_RESULTS) else 'will run'}")


In [ ]:
# =============================================================================
# BLOCK 1.5 (STAGE 1.5): ALWAYS-DEFINED FEATURE-EXTRACTION + ENTITY HELPERS
# Defined unconditionally so Stage 6 (multi-task eval) can call them even when
# Stages 2 and 3 are skipped because their output files already exist.
# =============================================================================

EXTRA_FEATURE_KEYS = [
    "F1_lookback_ratio", "F2_attention_sink", "F3_eigenscore_lite",
    "F4_icr_score", "F5_logit_lens_jsd", "F6_head_entropy",
    "F7_max_margin", "F8_token_rank", "F10_intra_dispersion",
]

# ---------------- entity helpers (used by Stage 2) ----------------
def delete_substrings(lst):
    subs = []; lst = list(set(lst))
    for s in lst:
        if any(s in o for o in lst if o != s): subs.append(s)
    for s in subs: lst.remove(s)
    return lst

def find_boundaries(text, words):
    out = []
    for word in words:
        ntext = text
        while True:
            start = ntext.find(word)
            if start == -1: break
            end = start + len(word) - 1
            while start > 0 and ntext[start-1] != " ": start -= 1
            while end < len(ntext)-1 and ntext[end+1] != " ": end += 1
            out.append("".join(ntext[i] for i in range(start, end+1)))
            ntext = ntext[end+1:]
    return out

def get_entities(text):
    LL = need_llm(); nlp = LL["nlp"]
    ents = list({str(e) for e in nlp(text).ents})
    ents = find_boundaries(text, ents)
    ents = delete_substrings(ents)
    out = []
    for i in range(len(text)):
        for e in ents:
            if text[i:].startswith(e): out.append((e, i))
    return out

def find_first_and_next_token(text, e, idx, input_id, prompt=""):
    LL = need_llm(); tokenizer = LL["tokenizer"]
    new_text = f"{text[:idx].strip()} {text[idx:].replace(e, e+' @', 1).strip()}"
    new_input_id = tokenizer(prompt + new_text.strip(), return_tensors="pt")["input_ids"].tolist()[0]
    for i in range(len(input_id[0])):
        if input_id[0][i] != new_input_id[i]: return []
    first_token = new_input_id[len(input_id[0])]
    at_cands = tokenizer("@", add_special_tokens=False)["input_ids"]
    at_cands += tokenizer(" @", add_special_tokens=False)["input_ids"]
    at_pos = None
    for at_tok in at_cands:
        try:
            at_pos = new_input_id.index(at_tok, len(input_id[0])); break
        except ValueError: continue
    if at_pos is None or at_pos >= len(new_input_id) - 1: return []
    next_token = new_input_id[at_pos + 1]
    entity_len = at_pos - len(input_id[0])
    last_id = new_input_id[at_pos + 1:]
    return [first_token, next_token, entity_len, last_id]


# ---------------- MIND 4 features (canonical + D_mean + V_last + H_last) ----------------
@torch.no_grad()
def extract_mind(text: str):
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    out = model(**enc, output_hidden_states=True, use_cache=False)
    hs = out.hidden_states
    last_per_layer = torch.stack([h[0, -1, :].float() for h in hs[1:]], dim=0)
    canonical = last_per_layer[-1].cpu().tolist()
    if last_per_layer.shape[0] < 2:
        D_mean = 0.0
    else:
        cos = F.cosine_similarity(last_per_layer[:-1], last_per_layer[1:], dim=1)
        D_mean = float((1.0 - cos).mean().item())
    mean_h = last_per_layer.mean(dim=0)
    V_last = float(((last_per_layer - mean_h) ** 2).sum(dim=1).mean().item())
    last_logits = out.logits[0, -1, :].float()
    p_last = torch.softmax(last_logits, dim=-1)
    H_last = float(-(p_last * p_last.clamp_min(1e-12).log()).sum().item())
    return canonical, D_mean, V_last, H_last


def per_step_entropy(scores):
    if not scores: return 0.0
    Hs = []
    for s in scores:
        p = torch.softmax(s[0].float(), dim=-1)
        Hs.append(float(-(p * p.clamp_min(1e-12).log()).sum().item()))
    return float(sum(Hs) / len(Hs))


# ---------------- 9 F features (F1, F2, F3, F4, F5, F6, F7, F8, F10) ----------------
def _jsd(p, q, eps=1e-12):
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    def _kl(a, b):
        return float((a * (a.clamp_min(eps).log() - b.clamp_min(eps).log())).sum().item())
    return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

_EXTRAS_CACHE = {"sink_ids": None}

def _ensure_extras_cache():
    if _EXTRAS_CACHE["sink_ids"] is not None: return
    LL = need_llm(); tokenizer = LL["tokenizer"]
    sink_strs = ["<|endoftext|>", "<s>", "</s>", "<|im_start|>", "<|im_end|>",
                  ".", ",", "!", "?", ";", ":", " .", " ,", " !", " ?", " ;", " :"]
    sink_ids = set()
    for s in sink_strs:
        try:
            for tid in tokenizer(s, add_special_tokens=False)["input_ids"]:
                sink_ids.add(int(tid))
        except Exception: continue
    for tid in [tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id]:
        if tid is not None: sink_ids.add(int(tid))
    _EXTRAS_CACHE["sink_ids"] = sink_ids


@torch.no_grad()
def extract_extras(text: str):
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    _ensure_extras_cache()
    SINK_IDS = _EXTRAS_CACHE["sink_ids"]
    idx = text.find(". ")
    prompt_text = text[: (idx + 2 if idx != -1 else len(text)//2)]
    enc = tokenizer(text.strip(), return_tensors="pt", truncation=True,
                    max_length=getattr(model.config, "max_position_embeddings", 4096)).to(model.device)
    input_ids = enc.input_ids; T = input_ids.shape[1]
    p_enc = tokenizer(prompt_text.strip(), return_tensors="pt", truncation=True,
                       max_length=getattr(model.config, "max_position_embeddings", 4096)).input_ids
    prompt_len = max(1, min(p_enc.shape[1], T - 1))
    out = model(**enc, output_hidden_states=True, output_attentions=True, use_cache=False)
    hs = out.hidden_states; attn = out.attentions; logits = out.logits

    last_layer_attn = attn[-1][0]
    last_pos = last_layer_attn[:, -1, :]
    mass_p = last_pos[:, :prompt_len].sum(dim=1).float()
    mass_g = last_pos[:, prompt_len:].sum(dim=1).float()
    F1 = float((mass_p / (mass_p + mass_g).clamp_min(1e-8)).mean().item())

    sink_mask = torch.zeros(T, dtype=torch.bool, device=input_ids.device)
    for i, tid in enumerate(input_ids[0].tolist()):
        if int(tid) in SINK_IDS: sink_mask[i] = True
    if sink_mask.any():
        sm = []
        for la in attn:
            a = la[0, :, -1, :].float()
            sm.append(float((a[:, sink_mask].sum(dim=1) / a.sum(dim=1).clamp_min(1e-8)).mean().item()))
        F2 = float(sum(sm) / len(sm))
    else:
        F2 = 0.0

    H_last = hs[-1][0].float()
    Hc = H_last - H_last.mean(dim=0, keepdim=True)
    if T >= 2:
        cov_t = (Hc @ Hc.t()) / max(T - 1, 1)
        cov_t = cov_t + 1e-3 * torch.eye(T, device=cov_t.device, dtype=cov_t.dtype)
        try:
            sgn, logabsdet = torch.slogdet(cov_t)
            F3 = float(logabsdet.item() if sgn > 0 else float("nan"))
        except Exception: F3 = float("nan")
    else: F3 = float("nan")

    ratios = []
    for l in range(1, len(hs)):
        h_prev = hs[l-1][0, -1, :].float(); h_curr = hs[l][0, -1, :].float()
        d = float(h_curr.norm().item())
        if d <= 1e-8: continue
        ratios.append(float((h_curr - h_prev).norm().item()) / d)
    F4 = float(sum(ratios) / max(len(ratios), 1))

    try:
        lm_w = model.get_output_embeddings().weight.float()
        ref = min(LOGIT_LENS_LAYER, len(hs) - 1)
        ph = hs[ref][0, -1, :].float()
        pe = torch.softmax(ph @ lm_w.t(), dim=-1)
        pf = torch.softmax(logits[0, -1, :].float(), dim=-1)
        F5 = _jsd(pe, pf)
    except Exception: F5 = float("nan")

    he = []
    for la in attn:
        a = la[0, :, -1, :].float().clamp_min(1e-12)
        a = a / a.sum(dim=1, keepdim=True).clamp_min(1e-12)
        he.append(float((-(a * a.log()).sum(dim=1)).mean().item()))
    F6 = float(sum(he) / max(len(he), 1))

    if T >= 2:
        probs = torch.softmax(logits[0, :-1, :].float(), dim=-1)
        top2 = probs.topk(2, dim=-1).values
        F7 = float((top2[:, 0] - top2[:, 1]).mean().item())
    else: F7 = 0.0

    if T >= 2:
        probs = torch.softmax(logits[0, :-1, :].float(), dim=-1)
        tgt = input_ids[0, 1:]; ranks = []
        si = probs.argsort(dim=-1, descending=True)
        for i in range(T - 1):
            pos = (si[i] == tgt[i]).nonzero(as_tuple=True)
            if len(pos[0]) > 0: ranks.append(float(pos[0][0].item() + 1))
        F8 = float(sum(ranks) / max(len(ranks), 1))
    else: F8 = 0.0

    if T >= 2:
        Hf = hs[-1][0].float()
        c = Hf.mean(dim=0, keepdim=True)
        F10 = float((Hf - c).pow(2).sum(dim=1).sqrt().mean().item())
    else: F10 = 0.0

    return {"F1_lookback_ratio": F1, "F2_attention_sink": F2,
            "F3_eigenscore_lite": F3, "F4_icr_score": F4,
            "F5_logit_lens_jsd": F5, "F6_head_entropy": F6,
            "F7_max_margin": F7, "F8_token_rank": F8,
            "F10_intra_dispersion": F10}


print("✓ feature helpers defined (extract_mind, extract_extras, get_entities, ...)")


In [ ]:
# =============================================================================
# BLOCK 2 (STAGE 2): MIND DATA GENERATION  (skip if dataset_full.json exists)
# =============================================================================
# STRICT-MODE GUARD: this notebook (02_all_variants.ipynb) does NOT do
# data generation. dataset_full.json MUST exist from running 01 first.
if not os.path.exists(PATH_DATASET_FULL):
    raise SystemExit(
        "\n[ERROR] " + PATH_DATASET_FULL + " not found.\n"
        "This notebook (02_all_variants.ipynb) does NOT generate data.\n"
        "Run 01_data_generation.ipynb first, then upload the JSON here.\n"
    )

if os.path.exists(PATH_DATASET_FULL):
    print(f"[SKIP STAGE 2] {PATH_DATASET_FULL} exists — loading.")
    _t0 = time.perf_counter()
    with open(PATH_DATASET_FULL, "r") as _f:
        records_full = json.load(_f)
    DIAG["stage_timings_sec"]["data_gen_skipped_load"] = round(time.perf_counter() - _t0, 2)
    n_h1 = sum(1 for r in records_full if r["label"] == 1)
    n_h0 = sum(1 for r in records_full if r["label"] == 0)
    DIAG["data_gen"] = {"loaded_from_disk": True, "n_hallucinated": n_h1,
                         "n_non_hallucinated": n_h0, "n_total": len(records_full)}
    print(f"  {len(records_full)} records  (y=1:{n_h1}  y=0:{n_h0})")
else:
    print(f"Generating MIND dataset -> {PATH_DATASET_FULL}")
    LL = need_llm()
    tokenizer, model = LL["tokenizer"], LL["model"]
    from datasets import load_dataset
    from nltk.tokenize import sent_tokenize
    from tqdm.auto import tqdm

    SKIP = {"overflow_prefix": 0, "no_token_match": 0, "overflow_suffix": 0,
             "model_knew": 0, "no_match_in_topk": 0, "empty_entity": 0,
             "duplicate_entity": 0, "exception": 0}

    def generate_sample(text, entity, idx, title):
        MAX_POS = getattr(model.config, "max_position_embeddings", 4096)
        HEADROOM = WINDOWS + 64
        enc = tokenizer(text[:idx].strip(), return_tensors="pt").to(model.device)
        input_ids = enc["input_ids"]; attn = enc["attention_mask"]
        if input_ids.shape[1] + HEADROOM >= MAX_POS:
            SKIP["overflow_prefix"] += 1; return None
        toks = find_first_and_next_token(text, entity, idx, input_ids.tolist())
        if not toks: SKIP["no_token_match"] += 1; return None
        first_, next_, entity_len, last_id = toks
        if input_ids.shape[1] + entity_len + len(last_id) + HEADROOM >= MAX_POS:
            SKIP["overflow_suffix"] += 1; return None
        gen = model.generate(input_ids, attention_mask=attn,
                             max_new_tokens=entity_len + WINDOWS,
                             return_dict_in_generate=True, output_scores=True,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
        if first_ in torch.topk(gen.scores[0], k=TOPK_FIRST_TOKEN).indices[0].tolist():
            SKIP["model_knew"] += 1; return None
        found_step = None
        for step, sc in enumerate(gen.scores):
            if next_ in torch.topk(sc, k=TOPK_FIRST_TOKEN).indices[0].tolist():
                found_step = step; break
        if found_step is None: SKIP["no_match_in_topk"] += 1; return None
        H_mean_hall = per_step_entropy(gen.scores[: found_step + 1])
        new_seq = gen.sequences[0, : input_ids.shape[1] + found_step].tolist()
        new_entity_ids = new_seq[input_ids.shape[1]:]
        hallucinated_ids = input_ids.tolist()[0] + new_entity_ids + last_id
        hallucinated_text = tokenizer.decode(hallucinated_ids, skip_special_tokens=True)
        new_entity = tokenizer.decode(new_entity_ids, skip_special_tokens=True).strip().lower()
        if not new_entity: SKIP["empty_entity"] += 1; return None
        if entity.lower() in new_entity or new_entity in text.lower():
            SKIP["duplicate_entity"] += 1; return None
        canon_h, D_h, V_h, _    = extract_mind(hallucinated_text)
        canon_o, D_o, V_o, H_o  = extract_mind(text)
        return {"text_hall": hallucinated_text, "entity_hall": new_entity,
                "canon_h": canon_h, "D_h": D_h, "V_h": V_h, "H_h": H_mean_hall,
                "text_orig": text, "entity_orig": entity,
                "canon_o": canon_o, "D_o": D_o, "V_o": V_o, "H_o": H_o,
                "title": title}

    wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    dataset_hall, dataset_non_hall = [], []
    processed = 0
    pbar = tqdm(total=TARGET_SAMPLES * 2, desc="samples")
    _t0 = time.perf_counter()
    for row in wiki:
        if len(dataset_hall) >= TARGET_SAMPLES and len(dataset_non_hall) >= TARGET_SAMPLES:
            break
        processed += 1
        try:
            sents = sent_tokenize(row["text"])
            if len(sents) < 2: continue
            text = " ".join(sents[:2]); title = row.get("title", "")
            ents = get_entities(text)
            if not ents: continue
            seen, ents_filt = set(), []
            for e, i in ents:
                if i == 0 or e.lower() in title.lower(): continue
                if i not in seen: seen.add(i); ents_filt.append((e, i))
            if not ents_filt: continue
            entity, char_idx = random.choice(ents_filt)
            result = generate_sample(text, entity, char_idx, title)
            if result is None: continue
            if len(dataset_hall) < TARGET_SAMPLES:
                dataset_hall.append({"label": 1, "text": result["text_hall"],
                                      "entity": result["entity_hall"],
                                      "embedding": result["canon_h"],
                                      "D_mean": result["D_h"], "V_last": result["V_h"],
                                      "H_mean": result["H_h"], "title": result["title"]})
                pbar.update(1)
            if len(dataset_non_hall) < TARGET_SAMPLES:
                dataset_non_hall.append({"label": 0, "text": result["text_orig"],
                                          "entity": result["entity_orig"],
                                          "embedding": result["canon_o"],
                                          "D_mean": result["D_o"], "V_last": result["V_o"],
                                          "H_mean": result["H_o"], "title": result["title"]})
                pbar.update(1)
            pbar.set_postfix(H1=len(dataset_hall), H0=len(dataset_non_hall), proc=processed)
            if (len(dataset_hall) + len(dataset_non_hall)) % 200 == 0:
                with open(PATH_CHECKPOINT, "w") as _f:
                    json.dump(dataset_hall + dataset_non_hall, _f)
        except Exception:
            SKIP["exception"] += 1; continue
        if processed % 100 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    pbar.close()
    DIAG["stage_timings_sec"]["data_gen"] = round(time.perf_counter() - _t0, 2)
    records_full = dataset_hall + dataset_non_hall
    with open(PATH_DATASET_FULL, "w") as _f:
        json.dump(records_full, _f)
    DIAG["data_gen"] = {
        "loaded_from_disk": False, "wiki_articles_processed": processed,
        "n_hallucinated": len(dataset_hall), "n_non_hallucinated": len(dataset_non_hall),
        "n_total": len(records_full), "skip_counters": dict(SKIP),
        "example_hall":     (dataset_hall[0]["text"][:280]     if dataset_hall else None),
        "example_non_hall": (dataset_non_hall[0]["text"][:280] if dataset_non_hall else None),
    }
    print(f"\n✓ wrote {PATH_DATASET_FULL}  ({os.path.getsize(PATH_DATASET_FULL)/1e6:.2f} MB)")
    print(f"  data_gen time: {DIAG['stage_timings_sec']['data_gen']} s   skip: {SKIP}")


In [ ]:
# =============================================================================
# BLOCK 3 (STAGE 3): COMPUTE F1..F10 per record  (skip if features file exists)
# =============================================================================
if os.path.exists(PATH_DATASET_FEATURES):
    print(f"[SKIP STAGE 3] {PATH_DATASET_FEATURES} exists — loading.")
    _t0 = time.perf_counter()
    with open(PATH_DATASET_FEATURES, "r") as _f:
        records_full = json.load(_f)
    DIAG["stage_timings_sec"]["features_skipped_load"] = round(time.perf_counter() - _t0, 2)
    print(f"  {len(records_full)} records, hidden_dim {len(records_full[0]['embedding'])}")
else:
    print(f"Computing F1..F10 for {len(records_full)} records ...")
    need_llm()
    _t0 = time.perf_counter()
    failures = 0
    accum = {k: [] for k in EXTRA_FEATURE_KEYS}
    from tqdm.auto import tqdm
    for i, r in enumerate(tqdm(records_full, desc="extract_extras")):
        try:
            extras = extract_extras(r["text"])
        except Exception as e:
            failures += 1
            # Print the first failure loudly so silent fall-back (all-zeros)
            # never goes unnoticed — this is exactly the bug we hit on
            # Qwen2.5 with default SDPA attention.
            if failures == 1:
                import traceback
                print(f"\n[!!] FIRST extract_extras FAILURE at row {i}:")
                print(f"     {type(e).__name__}: {e}")
                traceback.print_exc()
                print(f"     (further failures silenced — count tracked in failures)\n")
            DIAG["errors"].append(f"extract_extras row {i}: {type(e).__name__}: {e}")
            extras = {k: 0.0 for k in EXTRA_FEATURE_KEYS}
        for k, v in extras.items():
            r[k] = v; accum[k].append(v)
    DIAG["stage_timings_sec"]["feature_extraction"] = round(time.perf_counter() - _t0, 2)
    DIAG["n_feature_failures"] = failures
    DIAG["feature_stats"] = {k: {
        "mean": float(np.nanmean(v)) if v else float("nan"),
        "std":  float(np.nanstd(v))  if v else float("nan"),
        "min":  float(np.nanmin(v))  if v else float("nan"),
        "max":  float(np.nanmax(v))  if v else float("nan"),
    } for k, v in accum.items()}
    with open(PATH_DATASET_FEATURES, "w") as _f:
        json.dump(records_full, _f)
    print(f"\n✓ wrote {PATH_DATASET_FEATURES}  ({os.path.getsize(PATH_DATASET_FEATURES)/1e6:.2f} MB)")
    print(f"  feature_extraction: {DIAG['stage_timings_sec']['feature_extraction']} s   failures: {failures}")
    for k, st in DIAG["feature_stats"].items():
        print(f"   {k:25s} mean={st['mean']:.4f}  std={st['std']:.4f}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
# =============================================================================
# BLOCK 4 (STAGE 4): TRAIN 12 MLP VARIANTS + WIKIPEDIA HELD-OUT EVAL
# All variants share the same 80/20 split + same architecture + same hyperparams
# → results are directly comparable.
# =============================================================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, confusion_matrix, brier_score_loss)

ALL_SCALAR_KEYS = ["D_mean", "V_last", "H_mean"] + EXTRA_FEATURE_KEYS

VARIANTS = {
    "A": [],
    "B": ["D_mean"],
    "C": ["V_last"],
    "D": ["H_mean"],
    "E": ["D_mean", "V_last", "H_mean"],
    "F": ["F1_lookback_ratio"],
    "G": ["F5_logit_lens_jsd"],
    "H": ["F7_max_margin"],
    "I": ["F1_lookback_ratio", "F5_logit_lens_jsd", "F7_max_margin"],
    "J": ["F1_lookback_ratio", "F2_attention_sink", "F3_eigenscore_lite",
          "F4_icr_score", "F5_logit_lens_jsd", "F6_head_entropy",
          "F7_max_margin", "F8_token_rank", "F10_intra_dispersion"],
    "K": ["D_mean", "V_last", "H_mean",
          "F1_lookback_ratio", "F5_logit_lens_jsd", "F7_max_margin"],
    "L": ALL_SCALAR_KEYS,
}

# In-memory MLPs + scalers for Stage 6 (multi-task eval)
TRAINED = {}

# 80/20 split with seed=42 — IDENTICAL across all variants
random.seed(SEED)
records_shuf = list(records_full)
random.shuffle(records_shuf)
split_idx = int((1.0 - TEST_FRAC) * len(records_shuf))
train_recs = records_shuf[:split_idx]
test_recs  = records_shuf[split_idx:]
print(f"Split: train {len(train_recs)}   test {len(test_recs)}")
hidden_dim = len(records_full[0]["embedding"])


class MINDPlusClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 2),
        )
    def forward(self, x): return self.layers(x)


class _FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def _nan_fill(arr):
    # column-mean fill for any NaN (F3 sometimes returns NaN on T<2)
    if not np.isnan(arr).any(): return arr
    col_means = np.nanmean(arr, axis=0)
    inds = np.where(np.isnan(arr))
    arr[inds] = np.take(col_means, inds[1])
    return arr


def build_X_y(recs, keys, scaler=None, fit=False):
    canon = np.array([r["embedding"] for r in recs], dtype=np.float32)
    if keys:
        scalars = np.array([[r.get(k, 0.0) for k in keys] for r in recs], dtype=np.float32)
        scalars = _nan_fill(scalars)
        if fit: scaler = StandardScaler().fit(scalars)
        sz = scaler.transform(scalars).astype(np.float32)
        X = np.concatenate([canon, sz], axis=1)
    else:
        X = canon
    y = np.array([r["label"] for r in recs], dtype=np.int64)
    return X, y, scaler


def _safe_auc(y, pr):
    try: return float(roc_auc_score(y, pr)) if len(set(y.tolist())) > 1 else float("nan")
    except Exception: return float("nan")


def train_one_variant(letter, keys):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    X_tr, y_tr, scaler = build_X_y(train_recs, keys, fit=True)
    X_te, y_te, _      = build_X_y(test_recs,  keys, scaler=scaler, fit=False)
    input_dim = X_tr.shape[1]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mlp = MINDPlusClassifier(input_dim).to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(mlp.parameters(), lr=LR, weight_decay=WD)
    tl = DataLoader(_FeatureDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
    te = DataLoader(_FeatureDataset(X_te, y_te), batch_size=BATCH, shuffle=False)

    best_acc = 0.0; epoch_hist = []; best_state = None
    for ep in range(EPOCHS):
        mlp.train(); el = 0.0
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); lg = mlp(x); l = loss_fn(lg, y); l.backward(); opt.step()
            el += float(l.item())
        mlp.eval(); c = t = 0
        with torch.no_grad():
            for x, y in te:
                x, y = x.to(device), y.to(device)
                c += (mlp(x).argmax(1) == y).sum().item(); t += y.size(0)
        a = c / max(t, 1)
        epoch_hist.append({"epoch": ep+1,
                            "train_loss": round(el / max(len(tl), 1), 4),
                            "test_acc": round(a, 4)})
        if a > best_acc:
            best_acc = a; best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}

    if best_state is not None:
        ck = {"model_state": best_state, "input_dim": input_dim,
               "variant": letter, "scalar_keys": keys}
        if keys: ck["scaler_mean"] = scaler.mean_; ck["scaler_scale"] = scaler.scale_
        torch.save(ck, f"{MODEL_TAG}_variant_{letter}_best.pth")
        mlp.load_state_dict(best_state)

    TRAINED[letter] = {"mlp": mlp,
                        "scaler": scaler if keys else None,
                        "keys": list(keys),
                        "input_dim": int(input_dim)}

    # Wikipedia held-out
    mlp.eval(); ap, ay, apr = [], [], []
    with torch.no_grad():
        for x, y in te:
            x = x.to(device); lg = mlp(x); prob = torch.softmax(lg, dim=1)[:, 1]
            ap.extend(lg.argmax(1).cpu().numpy()); ay.extend(y.numpy())
            apr.extend(prob.cpu().numpy())
    ap = np.array(ap); ay = np.array(ay); apr = np.array(apr)
    acc = float(accuracy_score(ay, ap)) if len(ay) else float("nan")
    prec, rec, f1, _ = precision_recall_fscore_support(ay, ap, average="binary", zero_division=0)
    auc = _safe_auc(ay, apr)
    brier = float(brier_score_loss(ay, apr)) if len(ay) else float("nan")
    cm = confusion_matrix(ay, ap, labels=[0, 1])
    return {
        "feature_keys": keys, "input_dim": int(input_dim),
        "mlp_best_acc": float(best_acc), "mlp_epoch_history": epoch_hist,
        "wikipedia_eval": {
            "n": int(len(ay)), "accuracy": acc, "precision": float(prec),
            "recall": float(rec), "f1": float(f1), "auc_roc": auc, "brier": brier,
            "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]),
                   "fn": int(cm[1,0]), "tp": int(cm[1,1])},
            "label_distribution_test": {"y=0": int((ay==0).sum()), "y=1": int((ay==1).sum())},
            "pred_distribution_test":  {"p=0": int((ap==0).sum()), "p=1": int((ap==1).sum())},
            "prob_min":  round(float(apr.min()) if len(apr) else 0.0, 4),
            "prob_max":  round(float(apr.max()) if len(apr) else 0.0, 4),
            "prob_mean": round(float(apr.mean()) if len(apr) else 0.0, 4),
            "raw_y_pred_prob_first_50": [
                {"y": int(yy), "p": int(pp), "prob": round(float(pp2), 4)}
                for yy, pp, pp2 in zip(ay[:50], ap[:50], apr[:50])
            ],
        },
    }


print("Training 12 MLP variants ...")
_t0 = time.perf_counter()
variant_results = {}
for letter, keys in VARIANTS.items():
    variant_results[letter] = train_one_variant(letter, keys)
    m = variant_results[letter]["wikipedia_eval"]
    print(f"  variant {letter}  input_dim={variant_results[letter]['input_dim']:5d}  "
          f"AUROC={m['auc_roc']:.4f}  Acc={m['accuracy']:.4f}  F1={m['f1']:.4f}")
DIAG["stage_timings_sec"]["all_variants_train"] = round(time.perf_counter() - _t0, 2)
DIAG["variants"] = variant_results
print(f"\n✓ all 12 variants trained in {DIAG['stage_timings_sec']['all_variants_train']} s")


In [ ]:
# =============================================================================
# BLOCK 5 (STAGE 5): LOAD 7 MULTI-TASK EVAL DATASETS
# Same loaders as the uploaded project_smoke_gpt2.ipynb (with HF-namespace fix).
# =============================================================================
from datasets import load_dataset

print("Loading the 7 multi-task datasets ...")
def safe_load_first(*tries, subsample=None, label=""):
    last_err = None
    for i, fn in enumerate(tries):
        try:
            ds = fn()
            if subsample and len(ds) > subsample:
                ds = ds.shuffle(seed=SEED).select(range(subsample))
            print(f"  ✓ {label}: {len(ds)} samples  (loader #{i})")
            DIAG["downstream_datasets"][label] = {
                "loaded": True, "n": int(len(ds)),
                "loader_idx_used": i, "columns": list(ds.column_names),
            }
            return ds
        except Exception as e:
            last_err = e; continue
    print(f"  ✗ {label}: FAILED — {last_err}")
    DIAG["downstream_datasets"][label] = {"loaded": False, "error": str(last_err)}
    DIAG["errors"].append(f"dataset_load[{label}]: {last_err}")
    return None

_scale = min(0.2, max(0.05, TARGET_SAMPLES / 1000.0))
def _sub(n): return max(20, int(round(n * _scale)))
SUBSAMPLES = {"truthfulqa":     None if _scale >= 0.5 else _sub(200),
              "triviaqa":       _sub(500),
              "coqa":           _sub(500),
              "tydiqa":         _sub(500),
              "halueval_qa":    _sub(1000),
              "halueval_summ":  _sub(500),
              "halueval_dialog": _sub(1000)}
print(f"  (downstream scale cap = 0.2; effective _scale = {_scale:.2f})")
DIAG["downstream_datasets"]["subsamples_requested"] = dict(SUBSAMPLES)
print(f"Subsample scale = {_scale:.2f}; sizes = {SUBSAMPLES}")

LL = need_llm()
tokenizer, model = LL["tokenizer"], LL["model"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASETS = {}
DATASETS["truthfulqa"] = safe_load_first(
    lambda: load_dataset("truthfulqa/truthful_qa", "generation", split="validation"),
    lambda: load_dataset("truthful_qa", "generation", split="validation"),
    subsample=SUBSAMPLES["truthfulqa"], label="truthfulqa")
DATASETS["triviaqa"] = safe_load_first(
    lambda: load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation"),
    lambda: load_dataset("trivia_qa", "rc.nocontext", split="validation"),
    subsample=SUBSAMPLES["triviaqa"], label="triviaqa")
DATASETS["coqa"] = safe_load_first(
    lambda: load_dataset("stanfordnlp/coqa", split="validation"),
    subsample=SUBSAMPLES["coqa"], label="coqa")

def _load_tydi():
    ds = load_dataset("google-research-datasets/tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())
def _load_tydi_legacy():
    ds = load_dataset("tydiqa", "secondary_task", split="validation")
    return ds.filter(lambda x: "english" in x.get("id", "").lower())
DATASETS["tydiqa"] = safe_load_first(_load_tydi, _load_tydi_legacy,
    subsample=SUBSAMPLES["tydiqa"], label="tydiqa")

def _load_he(cfg):
    for cand in ("data", "train", "validation"):
        try: return load_dataset("pminervini/HaluEval", cfg, split=cand)
        except (ValueError, KeyError, FileNotFoundError): continue
    dd = load_dataset("pminervini/HaluEval", cfg); return dd[list(dd.keys())[0]]
for cfg, label in [("qa","halueval_qa"), ("summarization","halueval_summ"),
                   ("dialogue","halueval_dialog")]:
    DATASETS[label] = safe_load_first(
        lambda c=cfg: _load_he(c), subsample=SUBSAMPLES[label], label=label)

print()
for k, v in DATASETS.items():
    print(f"  {k:18s} -> {len(v) if v else 'unavailable'}")


In [ ]:
# =============================================================================
# BLOCK 5b (STAGE 5b): SENTENCE-TRANSFORMERS SCORER
# Same as uploaded notebook BLOCK 12 — MiniLM cosine for gold-vs-generation.
# =============================================================================
from sentence_transformers import SentenceTransformer

scorer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                              device=str(device))

def score_match(gen, golds, threshold=0.5):
    if not golds: return 0
    if isinstance(golds, str): golds = [golds]
    embs = scorer.encode([gen] + list(golds), convert_to_numpy=True,
                          normalize_embeddings=True)
    sims = embs[0] @ embs[1:].T
    return 0 if float(sims.max()) >= threshold else 1

print("✓ sentence-transformers scorer ready")


In [ ]:
# =============================================================================
# BLOCK 6 (STAGE 6): MULTI-TASK EVAL — ALL 12 VARIANTS x ALL 7 DATASETS
# Efficiency: ONE feature extraction per sample, all 12 MLPs share it.
# Same prompts + dataset handling as the uploaded notebook BLOCK 13.
# =============================================================================
from tqdm.auto import tqdm

@torch.no_grad()
def features_at_last_token_all(prompt, generation=""):
    text = (prompt + " " + generation).strip()
    canon, D_mean, V_last, H_last = extract_mind(text)
    extras = extract_extras(text)
    scalar_full = {"D_mean": D_mean, "V_last": V_last, "H_mean": H_last, **extras}
    return canon, scalar_full


@torch.no_grad()
def generate_short_answer(prompt, max_new=MAX_GEN_NEW):
    _max_pos = getattr(model.config, "max_position_embeddings", 4096)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=max(_max_pos - max_new, 256)).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new,
                          do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc.input_ids.shape[1]:],
                            skip_special_tokens=True).strip()


def classify_all_variants(canon, scalar_full):
    canon_arr = np.array(canon, dtype=np.float32)
    out = {}
    for letter, info in TRAINED.items():
        keys = info["keys"]
        if keys:
            scalars = np.array([[scalar_full.get(k, 0.0) for k in keys]], dtype=np.float32)
            if np.isnan(scalars).any():
                cm = info["scaler"].mean_
                inds = np.where(np.isnan(scalars))
                scalars[inds] = np.take(cm, inds[1])
            sz = info["scaler"].transform(scalars).astype(np.float32)[0]
            x = np.concatenate([canon_arr, sz])
        else:
            x = canon_arr
        x_t = torch.from_numpy(x).unsqueeze(0).to(device)
        lg = info["mlp"](x_t)
        prob = float(torch.softmax(lg, dim=1)[0, 1].item())
        pred = int(lg.argmax(1).item())
        out[letter] = (pred, prob)
    return out


def _compute_metrics(y, p, pr):
    y = np.asarray(y); p = np.asarray(p); pr = np.asarray(pr)
    if len(y) == 0: return {"n": 0}
    acc = float(accuracy_score(y, p))
    prec, rec, f1, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    try: auc = float(roc_auc_score(y, pr)) if len(set(y.tolist())) > 1 else float("nan")
    except ValueError: auc = float("nan")
    try: brier = float(brier_score_loss(y, pr))
    except ValueError: brier = float("nan")
    cm = confusion_matrix(y, p, labels=[0,1])
    return {"n": int(len(y)), "accuracy": acc,
            "precision": float(prec), "recall": float(rec),
            "f1": float(f1), "auc_roc": auc, "brier": brier,
            "cm": {"tn": int(cm[0,0]), "fp": int(cm[0,1]),
                   "fn": int(cm[1,0]), "tp": int(cm[1,1])},
            "label_distribution": {"y=0": int((y==0).sum()), "y=1": int((y==1).sum())},
            "pred_distribution":  {"p=0": int((p==0).sum()), "p=1": int((p==1).sum())},
            "prob_min":  round(float(pr.min()) if len(pr) else 0.0, 4),
            "prob_max":  round(float(pr.max()) if len(pr) else 0.0, 4),
            "prob_mean": round(float(pr.mean()) if len(pr) else 0.0, 4)}


def _eval_open_qa(ds, prompt_fn, gold_fn, dsname):
    pv = {l: {"y": [], "p": [], "pr": []} for l in TRAINED.keys()}
    fails = 0
    for s in tqdm(ds, desc=dsname):
        try:
            prompt = prompt_fn(s)
            gen = generate_short_answer(prompt, max_new=MAX_GEN_NEW)
            gold = gold_fn(s)
            label = score_match(gen, gold)
            canon, scalar_full = features_at_last_token_all(prompt, gen)
            preds = classify_all_variants(canon, scalar_full)
            for l, (pp, prob) in preds.items():
                pv[l]["y"].append(label); pv[l]["p"].append(pp); pv[l]["pr"].append(prob)
        except Exception:
            fails += 1; continue
    return pv, fails


def _eval_halueval(ds, prompt_fn, right_key, wrong_key, dsname):
    pv = {l: {"y": [], "p": [], "pr": []} for l in TRAINED.keys()}
    fails = 0
    for s in tqdm(ds, desc=dsname):
        try:
            prompt = prompt_fn(s)
            for ak, gl in [(right_key, 0), (wrong_key, 1)]:
                ans = s[ak]
                canon, scalar_full = features_at_last_token_all(prompt, ans)
                preds = classify_all_variants(canon, scalar_full)
                for l, (pp, prob) in preds.items():
                    pv[l]["y"].append(gl); pv[l]["p"].append(pp); pv[l]["pr"].append(prob)
        except Exception:
            fails += 1; continue
    return pv, fails


def _free_cuda():
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()


multitask_all = {}; fail_per_dataset = {}
_t0 = time.perf_counter()

if DATASETS["truthfulqa"] is not None:
    pv, fails = _eval_open_qa(DATASETS["truthfulqa"],
        prompt_fn=lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        gold_fn=lambda s: s.get("correct_answers", []),
        dsname="truthfulqa")
    multitask_all["truthfulqa"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["truthfulqa"] = fails; _free_cuda()

if DATASETS["triviaqa"] is not None:
    pv, fails = _eval_open_qa(DATASETS["triviaqa"],
        prompt_fn=lambda s: f"Answer the question concisely. Q: {s['question']} A:",
        gold_fn=lambda s: list(s["answer"].get("aliases", [])) + [s["answer"].get("value", "")],
        dsname="triviaqa")
    multitask_all["triviaqa"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["triviaqa"] = fails; _free_cuda()

if DATASETS["coqa"] is not None:
    def _coqa_pr(s):
        ctx = s["story"][:600]
        q = s["questions"][0] if s["questions"] else ""
        return f"Context: {ctx} Q: {q} A:"
    def _coqa_g(s):
        a = s["answers"]; return a["input_text"][0] if a["input_text"] else ""
    pv, fails = _eval_open_qa(DATASETS["coqa"], _coqa_pr, _coqa_g, "coqa")
    multitask_all["coqa"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["coqa"] = fails; _free_cuda()

if DATASETS["tydiqa"] is not None:
    pv, fails = _eval_open_qa(DATASETS["tydiqa"],
        prompt_fn=lambda s: f"Context: {s['context'][:600]} Q: {s['question']} A:",
        gold_fn=lambda s: s.get("answers", {}).get("text", []),
        dsname="tydiqa")
    multitask_all["tydiqa"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["tydiqa"] = fails; _free_cuda()

if DATASETS["halueval_qa"] is not None:
    pv, fails = _eval_halueval(DATASETS["halueval_qa"],
        prompt_fn=lambda s: f"Context: {s['knowledge'][:400]} Q: {s['question']} A:",
        right_key="right_answer", wrong_key="hallucinated_answer", dsname="halueval_qa")
    multitask_all["halueval_qa"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["halueval_qa"] = fails; _free_cuda()

if DATASETS["halueval_summ"] is not None:
    pv, fails = _eval_halueval(DATASETS["halueval_summ"],
        prompt_fn=lambda s: f"{s['document'][:500]} Summary:",
        right_key="right_summary", wrong_key="hallucinated_summary", dsname="halueval_summ")
    multitask_all["halueval_summ"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["halueval_summ"] = fails; _free_cuda()

if DATASETS["halueval_dialog"] is not None:
    pv, fails = _eval_halueval(DATASETS["halueval_dialog"],
        prompt_fn=lambda s: f"Knowledge: {s['knowledge'][:300]}\nDialogue: {s['dialogue_history'][:300]}\n[Assistant]:",
        right_key="right_response", wrong_key="hallucinated_response", dsname="halueval_dialog")
    multitask_all["halueval_dialog"] = {l: _compute_metrics(d["y"], d["p"], d["pr"]) for l, d in pv.items()}
    fail_per_dataset["halueval_dialog"] = fails

DIAG["stage_timings_sec"]["multitask_eval"] = round(time.perf_counter() - _t0, 2)

for letter in TRAINED.keys():
    DIAG["variants"][letter]["multitask"] = {
        ds_name: multitask_all[ds_name][letter] for ds_name in multitask_all.keys()
    }
DIAG["downstream_datasets"]["n_eval_failures_per_dataset"] = fail_per_dataset

print(f"\n* multi-task eval done in {DIAG['stage_timings_sec']['multitask_eval']} s")
print(f"  Datasets evaluated: {list(multitask_all.keys())}")
print(f"  Eval failures: {fail_per_dataset}")


In [ ]:
# =============================================================================
# BLOCK 7 (STAGE 7): CONSOLIDATED RESULTS DUMP
# Single JSON with env, config, data-gen stats, feature stats, and for each
# of 12 variants: feature_keys, input_dim, wikipedia_eval, multitask{ds: ...}
# =============================================================================
DIAG["timestamp_utc"] = (datetime.datetime.now(datetime.timezone.utc)
                          .replace(microsecond=0).isoformat().replace("+00:00", "Z"))
DIAG["host"] = ("kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ
                else ("colab" if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
                      else "local"))
DIAG["stage_timings_sec"]["total"] = round(sum(DIAG["stage_timings_sec"].values()), 2)

with open(PATH_RESULTS, "w") as f:
    json.dump(DIAG, f, indent=2, default=str)
print(f"\n* wrote {PATH_RESULTS}  ({os.path.getsize(PATH_RESULTS)/1024:.1f} KB)")

# Wikipedia summary
print("\n" + "=" * 90)
print(f"WIKIPEDIA HELD-OUT  -  {MODEL_TAG}")
print("=" * 90)
print(f"{'Var':4s} {'Features':50s} {'AUROC':>8s} {'Acc':>7s} {'F1':>7s} {'Brier':>7s}")
print("-" * 90)
for letter, vd in DIAG["variants"].items():
    m = vd["wikipedia_eval"]
    feats = ", ".join(vd["feature_keys"]) if vd["feature_keys"] else "(canonical only)"
    feats = (feats[:48] + "..") if len(feats) > 50 else feats
    print(f"{letter:4s} {feats:50s} {m['auc_roc']:8.4f} {m['accuracy']:7.4f} "
          f"{m['f1']:7.4f} {m['brier']:7.4f}")
print("=" * 90)

# 12 x 7 AUROC matrix
ds_order = ["truthfulqa", "triviaqa", "coqa", "tydiqa",
             "halueval_qa", "halueval_summ", "halueval_dialog"]
ds_short = {"truthfulqa": "TruthQA", "triviaqa": "TrivQA",
             "coqa": "CoQA", "tydiqa": "TydiQA",
             "halueval_qa": "HE-QA", "halueval_summ": "HE-Sum",
             "halueval_dialog": "HE-Dial"}
print("\n" + "=" * 90)
print(f"MULTI-TASK AUROC MATRIX  -  {MODEL_TAG}  (rows=variants, cols=datasets)")
print("=" * 90)
header = f"{'Var':4s} "
for ds in ds_order: header += f"{ds_short[ds]:>9s} "
print(header)
print("-" * 90)
for letter in DIAG["variants"].keys():
    mt = DIAG["variants"][letter].get("multitask", {})
    row = f"{letter:4s} "
    for ds in ds_order:
        m = mt.get(ds, {})
        if m and "auc_roc" in m:
            row += f"{m['auc_roc']:>9.3f} "
        else:
            row += f"{'-':>9s} "
    print(row)
print("=" * 90)

print("\nTimings:")
for k, v in DIAG["stage_timings_sec"].items():
    print(f"  {k:30s} {v:>8.2f} s")
print(f"\nFiles on disk:")
for p in [PATH_DATASET_FULL, PATH_DATASET_FEATURES, PATH_RESULTS]:
    if os.path.exists(p):
        print(f"  * {p}   ({os.path.getsize(p)/1e6:.2f} MB)")
print(f"  * {MODEL_TAG}_variant_<A..L>_best.pth   (12 files)")
print(f"\nPaste {PATH_RESULTS} back to the assistant.")
